# 05 - Train the five-second production model

This notebook mirrors the production window-training pipeline. It uses one row per five-second window, keeps all windows from a track in the same split, and trains the ensemble used by the Streamlit app. This prevents overlapping windows from the same song leaking into both training and testing.

## 1. Load the window feature table

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.train import build_ensemble, RANDOM_STATE
FEATURES_PATH = ROOT / 'data' / 'processed' / 'window_features.csv'
table = pd.read_csv(FEATURES_PATH)
table.head(), table.shape, table.groupby('label').size()

## 2. Split by track, not by window

In [ ]:
feature_columns = [
    column for column in table.columns
    if column not in {'track_id', 'label', 'window_start_seconds'}
]
X = table[feature_columns].to_numpy()
y = (table['label'] == 'ai_generated').astype('int8').to_numpy()
groups = table['track_id'].to_numpy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_indices, test_indices = next(splitter.split(X, y, groups))
X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]
print('train tracks:', table.iloc[train_indices]['track_id'].nunique())
print('test tracks:', table.iloc[test_indices]['track_id'].nunique())

## 3. Edit hyperparameters and train

In [ ]:
model = build_ensemble(len(feature_columns))
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)[:, 1]
predictions = (probabilities >= 0.5).astype('int8')
print('trained windows:', len(y_train))

## 4. Evaluate the held-out tracks

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
print('accuracy:', accuracy_score(y_test, predictions))
print('ROC-AUC:', roc_auc_score(y_test, probabilities))
print(classification_report(y_test, predictions, target_names=['human', 'ai_generated']))

## 5. Inspect window errors and confidence

In [ ]:
evaluation = table.iloc[test_indices][['track_id', 'label', 'window_start_seconds']].copy()
evaluation['ai_probability'] = probabilities
evaluation['prediction'] = ['ai_generated' if value else 'human' for value in predictions]
evaluation['correct'] = evaluation['label'] == evaluation['prediction']
evaluation.sort_values('ai_probability').head(10)